# Лабораторная работа 1.4. Метод вращений (Якоби)

**Задание.** Реализовать метод вращений в виде программы, задавая в качестве входных данных матрицу
и точность вычислений. Используя разработанное ПО, найти собственные значения и собственные векторы
симметрических матриц. Проанализировать зависимость погрешности вычислений от числа итераций.

$$
A = \begin{pmatrix} 2 & 8 & 7 \\ 8 & 2 & 7 \\ 7 & 7 & -8 \end{pmatrix}
$$

## Теория

Для симметрической матрицы $A$ существует ортогональная матрица $U$ такая, что
$U^T A U = \Lambda = \mathrm{diag}(\lambda_1, \dots, \lambda_n)$. Столбцы $U$ — собственные векторы.

Метод вращений строит $U$ как произведение элементарных вращений. На $k$-й итерации:

1. выбирается максимальный по модулю внедиагональный элемент $a_{ij}^{(k)}$, $i < j$;
2. строится матрица вращения $U^{(k)}$: единичная, кроме
   $u_{ii} = u_{jj} = \cos\varphi$, $\; u_{ij} = -\sin\varphi$, $\; u_{ji} = \sin\varphi$;
3. угол выбирается так, чтобы обнулить $a_{ij}^{(k+1)}$:

$$\varphi^{(k)} = \frac12 \operatorname{arctg} \frac{2 a_{ij}^{(k)}}{a_{ii}^{(k)} - a_{jj}^{(k)}},
\qquad \text{при } a_{ii}^{(k)} = a_{jj}^{(k)}:\ \varphi^{(k)} = \frac{\pi}{4};$$

4. $A^{(k+1)} = U^{(k)T} A^{(k)} U^{(k)}$.

**Критерий окончания** — малость суммы квадратов внедиагональных элементов:

$$t(A^{(k)}) = \left( \sum_{i<j} \left(a_{ij}^{(k)}\right)^2 \right)^{1/2} \le \varepsilon.$$

Тогда $\lambda_i \approx a_{ii}^{(k)}$, а собственные векторы — столбцы матрицы $U = U^{(0)} U^{(1)} \cdots U^{(k-1)}$.

Каждое вращение уменьшает $t^2$ ровно на $2\left(a_{ij}^{(k)}\right)^2 \ge \frac{2}{n(n-1)}\, t^2$,
поэтому $t(A^{(k)})$ убывает не медленнее геометрической прогрессии.
Вблизи решения сходимость становится квадратичной.

## Реализация

In [1]:
import math


def off_norm(A):
    """t(A) — корень из суммы квадратов наддиагональных элементов."""
    n = len(A)
    return math.sqrt(sum(A[i][j] ** 2 for i in range(n) for j in range(i + 1, n)))


def matmul(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(len(B))) for j in range(len(B[0]))]
            for i in range(len(A))]


def transpose(A):
    return [list(row) for row in zip(*A)]


def jacobi_rotation(A, eps, max_iter=1000):
    """Метод вращений Якоби для симметрической матрицы.

    Возвращает (собственные значения, матрица собственных векторов по столбцам,
                число итераций, история t(A^(k)), история диагоналей A^(k)).
    """
    n = len(A)
    for i in range(n):
        for j in range(i + 1, n):
            if abs(A[i][j] - A[j][i]) > 1e-12:
                raise ValueError("Матрица не симметрическая")

    A = [list(map(float, row)) for row in A]
    V = [[float(i == j) for j in range(n)] for i in range(n)]
    history = [off_norm(A)]
    diags = [[A[i][i] for i in range(n)]]

    k = 0
    while history[-1] > eps:
        if k >= max_iter:
            raise RuntimeError("Метод вращений не сошёлся за max_iter итераций")
        # максимальный по модулю внедиагональный элемент
        i, j = max(((p, q) for p in range(n) for q in range(p + 1, n)),
                   key=lambda pq: abs(A[pq[0]][pq[1]]))
        if A[i][i] == A[j][j]:
            phi = math.pi / 4
        else:
            phi = 0.5 * math.atan(2 * A[i][j] / (A[i][i] - A[j][j]))

        U = [[float(p == q) for q in range(n)] for p in range(n)]
        c, s = math.cos(phi), math.sin(phi)
        U[i][i], U[j][j] = c, c
        U[i][j], U[j][i] = -s, s

        A = matmul(matmul(transpose(U), A), U)
        V = matmul(V, U)
        k += 1
        history.append(off_norm(A))
        diags.append([A[i][i] for i in range(n)])

    return [A[i][i] for i in range(n)], V, k, history, diags


def show(M, name=None, prec=6):
    if name:
        print(f"{name} =")
    for row in M:
        print("  [" + " ".join(f"{(0.0 if abs(v) < 1e-12 else v):{prec + 6}.{prec}f}" for v in row) + " ]")
    print()

## Исходные данные

In [2]:
A = [
    [2, 8,  7],
    [8, 2,  7],
    [7, 7, -8],
]
eps = 1e-6

show(A, "A", prec=0)
print(f"eps = {eps}")

A =
  [     2      8      7 ]
  [     8      2      7 ]
  [     7      7     -8 ]

eps = 1e-06


## Собственные значения и собственные векторы

In [3]:
lams, V, k, hist, _ = jacobi_rotation(A, eps)

print(f"Итераций: {k},  t(A^(k)) = {hist[-1]:.2e}\n")
for idx, lam in enumerate(lams, 1):
    v = [V[r][idx - 1] for r in range(len(A))]
    print(f"lambda_{idx} = {lam:12.6f}    x_{idx} = [" + ", ".join(f"{c:9.6f}" for c in v) + " ]")
print()
show(V, "U (собственные векторы по столбцам)")

Итераций: 2,  t(A^(k)) = 2.22e-15

lambda_1 =    14.379088    x_1 = [ 0.646663,  0.646663,  0.404542 ]
lambda_2 =    -6.000000    x_2 = [-0.707107,  0.707107,  0.000000 ]
lambda_3 =   -12.379088    x_3 = [-0.286054, -0.286054,  0.914519 ]

U (собственные векторы по столбцам) =
  [    0.646663    -0.707107    -0.286054 ]
  [    0.646663     0.707107    -0.286054 ]
  [    0.404542     0.000000     0.914519 ]



## Проверка

In [4]:
n = len(A)
print("Невязки ||A x_i - lambda_i x_i||:")
for idx in range(n):
    v = [V[r][idx] for r in range(n)]
    Av = [sum(A[r][c] * v[c] for c in range(n)) for r in range(n)]
    print(f"  i={idx + 1}: {max(abs(Av[r] - lams[idx] * v[r]) for r in range(n)):.2e}")

print("\nОртогональность: U^T U =")
show(matmul(transpose(V), V))

print(f"Сумма lambda = {sum(lams):.6f},  след A = {sum(A[i][i] for i in range(n))}")

Невязки ||A x_i - lambda_i x_i||:
  i=1: 1.78e-15
  i=2: 8.88e-16
  i=3: 1.33e-15

Ортогональность: U^T U =
  [    1.000000     0.000000     0.000000 ]
  [    0.000000     1.000000     0.000000 ]
  [    0.000000     0.000000     1.000000 ]

Сумма lambda = -4.000000,  след A = -4


## Анализ зависимости погрешности от числа итераций

Погрешность оценивается двумя способами:
- $t(A^{(k)})$ — внутренний критерий метода (сумма квадратов внедиагональных элементов);
- $\max_i |a_{ii}^{(k)} - \lambda_i|$ — фактическая погрешность собственных значений относительно
  эталона (решения с точностью $10^{-14}$).

In [5]:
def error_table(A):
    lams_ref, _, _, hist, diags = jacobi_rotation(A, 1e-14)
    lams_ref = sorted(lams_ref)
    print(f"{'k':>3} | {'t(A^(k))':>10} | {'t_k / t_(k-1)':>13} | {'max|a_ii - lambda_i|':>20}")
    print("-" * 57)
    for k, (t, d) in enumerate(zip(hist, diags)):
        ratio = f"{t / hist[k - 1]:13.3e}" if k > 0 else " " * 13
        err = max(abs(x - y) for x, y in zip(sorted(d), lams_ref))
        print(f"{k:3} | {t:10.3e} | {ratio} | {err:20.3e}")


def iterations_table(A):
    print(f"{'eps':>8} | {'итераций':>8} | {'t(A^(k))':>10}")
    print("-" * 33)
    for p in range(1, 13):
        e = 10.0 ** -p
        _, _, k_e, h_e, _ = jacobi_rotation(A, e)
        print(f"{e:8.0e} | {k_e:8} | {h_e[-1]:10.3e}")


error_table(A)

  k |   t(A^(k)) | t_k / t_(k-1) | max|a_ii - lambda_i|
---------------------------------------------------------
  0 |  1.273e+01 |               |            1.238e+01
  1 |  9.899e+00 |     7.778e-01 |            4.379e+00
  2 |  2.220e-15 |     2.243e-16 |            0.000e+00


**Матрица варианта сходится всего за 2 вращения — это следствие её структуры.**
Она симметрична относительно перестановки $x_1 \leftrightarrow x_2$, и $a_{11} = a_{22}$.
Первое вращение в плоскости $(1, 2)$ на угол $\varphi = \pi/4$ сразу выделяет точный собственный вектор
$(-1, 1, 0)^T/\sqrt2$ с $\lambda = -6$. Остаётся блок $2 \times 2$, а для него одно вращение всегда
даёт точную диагонализацию. Третья строка таблицы — это уже вращение на уровне машинного нуля.

Поэтому для анализа зависимости погрешности от числа итераций дополнительно возьмём
симметрическую матрицу $5 \times 5$ общего вида.

In [6]:
B = [
    [ 4, -3,  2,  1,  5],
    [-3,  6,  1, -2,  3],
    [ 2,  1, -5,  4,  2],
    [ 1, -2,  4,  7, -1],
    [ 5,  3,  2, -1,  1],
]
show(B, "B", prec=0)
error_table(B)

B =
  [     4     -3      2      1      5 ]
  [    -3      6      1     -2      3 ]
  [     2      1     -5      4      2 ]
  [     1     -2      4      7     -1 ]
  [     5      3      2     -1      1 ]

  k |   t(A^(k)) | t_k / t_(k-1) | max|a_ii - lambda_i|
---------------------------------------------------------
  0 |  8.602e+00 |               |            5.435e+00
  1 |  7.000e+00 |     8.137e-01 |            2.697e+00
  2 |  5.602e+00 |     8.002e-01 |            2.697e+00
  3 |  3.922e+00 |     7.001e-01 |            2.206e+00
  4 |  2.919e+00 |     7.443e-01 |            2.206e+00
  5 |  2.161e+00 |     7.402e-01 |            4.855e-01
  6 |  1.710e+00 |     7.914e-01 |            3.812e-01
  7 |  1.188e+00 |     6.949e-01 |            3.812e-01
  8 |  7.642e-01 |     6.431e-01 |            1.258e-01
  9 |  6.035e-01 |     7.897e-01 |            2.299e-02
 10 |  4.324e-01 |     7.166e-01 |            7.182e-03
 11 |  3.155e-01 |     7.297e-01 |            7.182e-03
 12 |  1.

In [7]:
print("Число итераций для разных eps (матрица B):\n")
iterations_table(B)

Число итераций для разных eps (матрица B):

     eps | итераций |   t(A^(k))
---------------------------------
   1e-01 |       14 |  3.682e-02
   1e-02 |       19 |  2.291e-03
   1e-03 |       21 |  8.795e-04
   1e-04 |       23 |  5.365e-05
   1e-05 |       24 |  2.075e-06
   1e-06 |       25 |  6.754e-07
   1e-07 |       28 |  4.964e-09
   1e-08 |       28 |  4.964e-09
   1e-09 |       29 |  2.934e-10
   1e-10 |       30 |  4.441e-11
   1e-11 |       31 |  6.638e-14
   1e-12 |       31 |  6.638e-14


## Проверка с помощью NumPy (необязательно)

In [8]:
try:
    import numpy as np
    w, Vn = np.linalg.eigh(np.array(A, float))
    print("lambda (numpy):", w)
    print("lambda (Якоби):", np.sort(lams))
    print("max |lambda - lambda_numpy| =", np.abs(np.sort(lams) - w).max())
    lams_B, _, _, _, _ = jacobi_rotation(B, eps)
    print("\nМатрица B: max |lambda - lambda_numpy| =",
          np.abs(np.sort(lams_B) - np.linalg.eigvalsh(np.array(B, float))).max())
except ImportError:
    print("NumPy не установлен — пропускаем сверку")

lambda (numpy): [-12.37908816  -6.          14.37908816]
lambda (Якоби): [-12.37908816  -6.          14.37908816]
max |lambda - lambda_numpy| = 2.6645352591003757e-15

Матрица B: max |lambda - lambda_numpy| = 1.545430450278218e-13


## Вывод

Реализован метод вращений Якоби с выбором максимального по модулю внедиагонального элемента.
Для заданной матрицы найдены собственные значения и ортонормированный базис из собственных векторов.
Невязки $\|Ax_i - \lambda_i x_i\|$ и проверка $U^T U = E$ подтверждают корректность.

**Зависимость погрешности от числа итераций.**

- Матрица варианта ($3 \times 3$) диагонализируется точно за 2 вращения благодаря своей симметрии
  ($a_{11} = a_{22}$, $a_{13} = a_{23}$), так что от $\varepsilon$ число итераций не зависит.
- Для матрицы $B$ общего вида ($5 \times 5$) на первых ~14 итерациях $t(A^{(k)})$ убывает линейно,
  с отношением $t_k / t_{k-1} \approx 0.7\text{–}0.8$. Это укладывается в теоретическую оценку
  $\sqrt{1 - 2/(n(n-1))} = \sqrt{0.8} \approx 0.89$.
- Затем внедиагональные элементы становятся малыми, отношение $t_k / t_{k-1}$ падает до $10^{-2}\text{–}10^{-6}$,
  и сходимость становится квадратичной. Чтобы достичь $\varepsilon = 10^{-1}$, нужно 14 итераций, а следующие
  10 порядков точности (до $10^{-11}$) добавляют всего 17 итераций.
- Погрешность собственных значений $\max|a_{ii} - \lambda_i|$ убывает значительно быстрее, чем $t(A^{(k)})$:
  она имеет порядок $t^2$. Например, при $t \approx 2 \cdot 10^{-3}$ погрешность уже $\approx 4 \cdot 10^{-7}$.
  Поэтому при остановке по $t \le \varepsilon$ собственные значения фактически получаются гораздо точнее,
  чем $\varepsilon$.